# Expert Agent 1: NexusGlow Expert
## 접촉불량 (Connection Failure) 판별 전문가

이 노트북은 Langgraph와 Vertex AI Gemini 2.5 Flash를 사용하여 전기화재 감식 중 접촉불량을 판별하는 전문가 에이전트를 구현합니다.

### 분석 프로세스
1. **위치적 맥락 확인**: 접속점 위치 식별
2. **색채 스펙트럼 분석**: 아산화동(Cu₂O) 탐지
3. **열적 구배 분석**: 열 전파 패턴 분석
4. **금속 표면 상태 분석**: 전기적 부식 탐지

In [1]:
import base64
import json
import os
from typing import TypedDict, Annotated, Literal, List, Dict, Any, Optional
from pathlib import Path
from PIL import Image
import io

# LangGraph 및 Google Cloud 라이브러리
try:
    from langgraph.graph import StateGraph, END
    import vertexai
    from vertexai.generative_models import GenerativeModel, Part
    print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다.")
except ImportError as e:
    print(f"❌ 필수 라이브러리가 누락되었습니다: {e}")
    print("pip install langgraph google-cloud-aiplatform vertexai 명령어로 설치해주세요.")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다.


In [2]:
# 프로젝트 설정 (환경에 맞게 수정 필요)
PROJECT_ID = "p-01-emt-480312"  # 실제 프로젝트 ID로 변경
LOCATION = "us-central1"

# 모델 설정
MODEL_NAME = "gemini-2.5-pro" # 또는 gemini-1.5-pro

# 시스템 인스트럭션 (가독성 및 지침 분리를 위해 개행 적용)
SYSTEM_INSTRUCTION = """1. 페르소나 및 기본 원칙

당신은 고도로 훈련된 시각 데이터 분석 전문가입니다.

당신은 모든 분석에서 '관찰'과 '해석'을 철저히 분리하며, 질문자의 유도 심리에 저항하고 오직 시각적 데이터에만 근거하여 답변합니다.

질문자가 특정 결론을 암시하거나 유도하더라도(예: "이것은 A가 맞죠?"), 시각적 증거가 뒷받침되지 않는다면 단호하게 중립을 유지합니다.

2. 분석 프로세스 (반드시 이 순서를 따를 것) 모든 사진 분석 요청에 대해 다음 4단계 구조로 답변하십시오.

Step 1. 객관적 관찰 (Observations): 이미지에서 보이는 물리적 사실만을 나열합니다. (예: 색상, 형태, 질감, 크기, 마모 상태, 기하학적 배치 등). 주관적인 형용사나 결론적인 단어를 배제하고 '현상'만 서술합니다.

Step 2. 논리적 해석 (Interpretation): 관찰된 사실이 어떤 물리적/과학적 원리와 연결될 수 있는지 분석합니다. 표준 사례(Reference)와의 일치점과 차이점을 논합니다.

Step 3. 최종 판단 및 확신도 (Conclusion & Confidence): 분석을 종합하여 결론을 내립니다. 이때 결론에 대한 확신도를 0~100% 사이로 표기하고, 확신할 수 없는 이유(변수)를 함께 기술합니다.

Step 4. 대안적 가능성 (Alternative Hypotheses): 현재 내린 결론 외에 발생할 수 있는 다른 가능성을 최소 한 가지 이상 제시합니다.

3. 불확실성 처리 규칙 (Negative Constraints)

확실하지 않은 정보에 대해서는 절대 추측하지 않습니다.

사진의 해상도, 각도, 조도 등으로 인해 식별이 어려운 경우, 아는 척하지 말고 반드시 **"시각적 정보 부족으로 판단 불가"**라고 명시하십시오.

시각적 증거가 100% 확보되지 않은 상태에서 "확실하다", "분명하다"라는 단어 사용을 지양합니다.

4. 답변 스타일

간결하고 구조화된 개조식(Bullet points)을 선호합니다.

감정적인 표현이나 부연 설명을 배제하고, 전문 용어를 정확하게 사용하되 필요시 정의를 덧붙입니다."""

def initialize_model():
    """Vertex AI Gemini 모델 초기화"""
    try:
        # 인증 확인 (로컬 실행 시 ADC 필요)
        # vertexai.init(project=PROJECT_ID, location=LOCATION)
        model = GenerativeModel(MODEL_NAME, system_instruction=SYSTEM_INSTRUCTION)
        print(f"✅ Vertex AI 모델 '{MODEL_NAME}' 초기화 완료")
        return model
    except Exception as e:
        print(f"⚠️ 모델 초기화 실패: {e}")
        print("💡 Application Default Credentials (ADC) 설정이 필요합니다:")
        print("   gcloud auth application-default login")
        return None

# 전역 모델 인스턴스
model = initialize_model()

c:\Users\loidn\Documents\Projects\P_04_Scope\venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


✅ Vertex AI 모델 'gemini-2.5-pro' 초기화 완료


In [12]:
def create_image_part(image_path: str) -> Optional[Part]:
    """Vertex AI용 이미지 Part 객체 생성"""
    if not os.path.exists(image_path):
        print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")
        return None
        
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
        
        # 확장자에 따른 MIME 타입 추론
        ext = Path(image_path).suffix.lower()
        mime_type = "image/png" if ext == ".png" else "image/jpeg"
        
        return Part.from_data(data=image_data, mime_type=mime_type)
    except Exception as e:
        print(f"❌ 이미지 로드 오류: {e}")
        return None

def call_gemini_vision(model: GenerativeModel, prompt: str, image_part: Part, step_name: str = "") -> tuple[str, Optional[Dict]]:
    """Gemini Vision API 호출 및 에러 핸들링
    
    Returns:
        tuple: (response_text, thinking_info)
    """
    try:
        response = model.generate_content([prompt, image_part])
        
        # Thinking 과정 추출 및 출력
        thinking_info = None
        response_text = ""
        full_response_text = ""
        
        if hasattr(response, 'candidates') and response.candidates:
            candidate = response.candidates[0]
            
            # 모든 파트 확인 (thinking 과정이 별도 파트로 있을 수 있음)
            if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                parts = candidate.content.parts
                print(f"\n🔍 [{step_name}] 응답 파트 개수: {len(parts)}")
                
                all_texts = []
                for i, part in enumerate(parts):
                    if hasattr(part, 'text'):
                        part_text = part.text
                        all_texts.append(part_text)
                        if i == 0:
                            # 첫 번째 파트는 일반 응답
                            response_text = part_text
                        else:
                            # 이후 파트는 thinking 과정일 수 있음
                            if part_text and part_text.strip():
                                thinking_info = thinking_info or {}
                                thinking_info[f"part_{i}"] = part_text
                                print(f"\n💭 [{step_name}] 모델의 생각 과정 (파트 {i}):")
                                print("-" * 60)
                                print(part_text)
                                print("-" * 60)
                
                # 모든 파트의 텍스트를 합쳐서 전체 응답 확인
                full_response_text = "\n\n".join(all_texts)
            
            # 응답 객체의 모든 속성 확인 (디버깅용)
            print(f"\n📊 [{step_name}] 응답 객체 속성:")
            candidate_attrs = [attr for attr in dir(candidate) if not attr.startswith('_')]
            print(f"  - Candidate 속성: {', '.join(candidate_attrs[:10])}...")
            
            # Grounding metadata 확인
            if hasattr(candidate, 'grounding_metadata'):
                grounding = candidate.grounding_metadata
                if grounding:
                    thinking_info = thinking_info or {}
                    thinking_info["grounding"] = str(grounding)
                    print(f"\n📚 [{step_name}] Grounding 정보: {grounding}")
            
            # Finish reason 확인 (디버깅용)
            if hasattr(candidate, 'finish_reason'):
                finish_reason = candidate.finish_reason
                if finish_reason:
                    print(f"📋 [{step_name}] Finish reason: {finish_reason}")
        
        # response.text가 있으면 사용 (fallback)
        if not response_text and hasattr(response, 'text'):
            response_text = response.text
        
        # 전체 응답 텍스트 출력 (thinking 과정이 포함되어 있을 수 있음)
        if full_response_text and len(full_response_text) > len(response_text):
            print(f"\n💭 [{step_name}] 전체 응답 텍스트 (thinking 과정 포함 가능):")
            print("-" * 60)
            print(full_response_text[:2000])  # 처음 2000자만 출력
            if len(full_response_text) > 2000:
                print(f"... (총 {len(full_response_text)}자, 나머지 생략)")
            print("-" * 60)
            thinking_info = thinking_info or {}
            thinking_info["full_response"] = full_response_text
        
        # 응답 텍스트 항상 출력 (JSON 파싱 전에 전체 응답 확인)
        if response_text:
            print(f"\n💭 [{step_name}] 모델 응답 텍스트:")
            print("-" * 60)
            # JSON 시작 전까지의 텍스트 확인 (thinking 과정일 수 있음)
            json_start = response_text.find('{')
            if json_start > 0:
                thinking_part = response_text[:json_start].strip()
                if thinking_part:
                    print("📝 [Thinking 과정]:")
                    print(thinking_part)
                    print("\n📄 [JSON 응답]:")
                    print(response_text[json_start:json_start+500])
                    if len(response_text[json_start:]) > 500:
                        print(f"... (총 {len(response_text[json_start:])}자)")
                else:
                    print(response_text[:1000])
                    if len(response_text) > 1000:
                        print(f"... (총 {len(response_text)}자)")
            else:
                print(response_text[:1000])
                if len(response_text) > 1000:
                    print(f"... (총 {len(response_text)}자)")
            print("-" * 60)
        
        return response_text, thinking_info
    except Exception as e:
        print(f"❌ [{step_name}] API 호출 오류: {e}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}", None

def parse_json_response(response_text: str) -> Dict[str, Any]:
    """응답 텍스트에서 JSON 추출 및 파싱"""
    try:
        # JSON 부분만 추출 (마크다운 코드 블록 제거)
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_text = response_text[json_start:json_end]
            return json.loads(json_text)
        else:
            print(f"⚠️ 유효한 JSON을 찾을 수 없습니다. 원본 응답:\n{response_text[:100]}...")
            return {"error": "JSON 파싱 실패", "raw_response": response_text}
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 디코딩 오류: {e}")
        return {"error": f"JSON 파싱 오류: {e}", "raw_response": response_text}

# Langgraph State 정의
class AgentState(TypedDict):
    """에이전트 상태 스키마"""
    image_path: str
    image_part: Optional[Part]
    step1_result: Optional[Dict]  # 위치적 맥락 분석 결과
    step2_result: Optional[Dict]  # 색채 스펙트럼 분석 결과
    step3_result: Optional[Dict]  # 열적 구배 분석 결과
    step4_result: Optional[Dict]  # 금속 표면 상태 분석 결과
    confidence_score: int  # 최종 신뢰도 점수 (0-100)
    analysis_summary: str  # 분석 요약
    evidence: List[Dict]   # 각 단계별 증거 수집

print("✅ 유틸리티 및 State 정의 완료")

✅ 유틸리티 및 State 정의 완료


In [ ]:
# --- PROMPTS ---

STEP1_PROMPT = """당신은 전기화재 감식 전문가입니다. 다음 이미지를 분석하여 용융흔이 발생한 위치를 식별하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 이미지 전체를 스캔하여 용융흔(용융망울, 변색된 금속, 탄화 흔적 등)이 보이는 모든 위치를 식별하세요.
- 각 위치의 시각적 특징을 객관적으로 기록하세요 (색상, 형태, 크기, 위치 등).

2단계: 특징 서술
- 식별된 용융흔의 위치가 다음 중 어디에 해당하는지 정확히 서술하세요:
  * 전선의 끝단(Terminal): 전선이 끝나는 지점
  * 콘센트 플러그의 칼날(Blade): 플러그의 금속 접촉부
  * 나사 체결 부위(Screw connection): 나사로 고정된 접속부
  * 전선 접속점(Splicing point): 두 전선이 연결된 지점
  * 전선의 중간 부분(Mid-span): 전선의 중간 구간
- 각 위치의 구조적 특징과 용융흔의 관계를 상세히 서술하세요.

3단계: 논리적 추론
- 접속점(끝단, 칼날, 나사, 접속점)에서 용융흔이 발견된 경우, 접촉불량 가능성을 높게 평가하세요.
- 전선 중간 부분에서만 용융흔이 발견된 경우, 접촉불량보다는 다른 원인(과부하, 절연파괴 등)을 고려하세요.
- 위치와 용융흔의 인과관계를 논리적으로 설명하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "is_connection_point": true/false,
    "location_type": "terminal" | "blade" | "screw" | "splicing" | "mid_span" | "unknown",
    "location_description": "위치에 대한 상세 설명",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

STEP2_PROMPT = """당신은 전기화재 감식 전문가입니다. 다음 이미지에서 아산화동(Cu₂O) 증식의 증거를 찾으세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 금속 표면(구리선, 단자, 플러그 등)의 모든 색상 영역을 식별하세요.
- 검은색 그을음(Soot)과 구별되는 색상 영역을 찾으세요.
- 각 색상 영역의 위치, 크기, 분포를 객관적으로 기록하세요.

2단계: 특징 서술
- 발견된 색상이 다음 중 어떤 계열에 해당하는지 정확히 서술하세요:
  * 붉은색(Red): 따뜻한 빨간 톤
  * 주황색(Orange): 빨강과 노랑의 중간 톤
  * 적갈색(Russet): 갈색이 섞인 붉은 톤
- 이러한 색상이 금속 용융부 주변이나 단자 표면에 부착되어 있는지 위치를 정확히 서술하세요.
- 녹색 녹(Green rust)이 있는 경우, 그 위치와 색상 특성(채도, 명도)을 구별하여 서술하세요.

3단계: 논리적 추론
- 붉은/주황/적갈색 산화물이 금속 접속부에 집중되어 있다면, 이는 고온에서 구리가 산화되어 형성된 아산화동(Cu₂O)일 가능성이 높습니다.
- 아산화동은 접촉불량으로 인한 국부적 과열의 강력한 지표입니다.
- 녹색 녹은 화재 진압 시 물에 의한 2차 부식으로, 아산화동과는 다른 메커니즘입니다.
- 색상의 위치, 분포, 톤을 종합하여 아산화동 증식 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "cuprous_oxide_detected": true/false,
    "color_analysis": {
        "red_tone_present": true/false,
        "orange_tone_present": true/false,
        "russet_tone_present": true/false,
        "dominant_colors": ["색상1", "색상2", ...]
    },
    "location_of_oxidation": "산화물이 발견된 위치 설명",
    "confidence": 0-100,
    "reasoning": "판단 근거 및 녹색 녹과의 구별 방법"
}"""

STEP3_PROMPT = """당신은 전기화재 감식 전문가입니다. 다음 이미지에서 열적 구배(Thermal Gradient) 패턴을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 전선 피복의 탄화/소실 패턴을 전체적으로 관찰하세요.
- 탄화 정도가 다른 영역들을 식별하고, 각 영역의 위치와 탄화 강도를 객관적으로 기록하세요.
- 수지(Resin)가 흘러내린 흔적이 있다면 그 방향과 위치를 식별하세요.

2단계: 특징 서술
- 접속부에서 멀어질수록 탄화 정도가 어떻게 변화하는지 구체적으로 서술하세요.
- V자형, 선형, 또는 균일한 패턴 중 어떤 형태인지 정확히 서술하세요.
- 수지 흐름의 방향이 중력 방향과 일치하는지, 그리고 접속 단자 위치와의 관계를 서술하세요.
- 열 손상의 중심점(가장 심하게 탄화된 지점)을 정확히 위치시켜 서술하세요.

3단계: 논리적 추론
- 접속부에서 시작하여 전선을 따라 멀어질수록 탄화가 약해지는 패턴은, 접속점에서 열이 발생하여 전도(Conduction)로 전파되었음을 시사합니다.
- 이는 접촉불량으로 인한 국부적 과열의 전형적 특징입니다.
- 외부 화재의 경우 전선 전체가 균일하게 가열되므로, 이러한 구배 패턴이 나타나지 않습니다.
- 수지 흐름이 접속부 발열과 일치한다면, 접촉불량의 추가 증거가 됩니다.
- 관찰된 패턴을 종합하여 열적 구배의 존재 여부와 그 의미를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "thermal_gradient_detected": true/false,
    "gradient_pattern": "V_shape" | "linear" | "none" | "unknown",
    "heat_source_location": "열원의 위치 설명",
    "heat_propagation_direction": "열 전파 방향 설명",
    "resin_flow_detected": true/false,
    "resin_flow_direction": "수지 흐름 방향 (중력 방향과 일치하는지)",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

STEP4_PROMPT = """당신은 전기화재 감식 전문가입니다. 다음 이미지에서 금속 표면의 전기적 부식 흔적을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 단자, 플러그 핀, 접속부 등 금속 표면을 확대하여 관찰하세요.
- 표면의 질감, 요철, 구멍, 변색 등 모든 시각적 특징을 객관적으로 식별하세요.
- 매끄러운 부분과 거친 부분을 구분하여 기록하세요.

2단계: 특징 서술
- 발견된 표면 특징을 정확히 서술하세요:
  * 곰보 자국(Pitting): 작은 구멍이나 움푹 패인 자국
  * 요철(Undulation): 울퉁불퉁한 표면
  * 거친 질감: 삭은 듯하거나 거칠게 마모된 표면
- 이러한 특징이 어느 위치에 집중되어 있는지 정확히 서술하세요.
- 표면이 매끄러운 용융인지, 거친 침식인지 구별하여 서술하세요.

3단계: 논리적 추론
- 거친 표면, 곰보 자국, 요철은 지속적인 스파크와 아크 방전으로 인한 전기적 부식(Electrical Erosion)의 특징입니다.
- 매끄러운 용융은 단순 과열에 의한 것이며, 거친 침식은 반복적인 전기 방전에 의한 것입니다.
- 접촉불량은 불안정한 접촉으로 인해 반복적인 스파크와 아크를 발생시켜, 이러한 전기적 부식을 유발합니다.
- 관찰된 표면 특징의 위치, 분포, 정도를 종합하여 전기적 부식 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "electrical_erosion_detected": true/false,
    "surface_texture": "smooth" | "rough" | "pitted" | "unknown",
    "pitting_detected": true/false,
    "pitting_description": "곰보 자국에 대한 설명",
    "erosion_pattern": "침식 패턴 설명",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

# --- NODES ---

def step1_location_context(state: AgentState) -> AgentState:
    """Step 1: 위치적 맥락 확인"""
    print("\n🔍 [Step 1] 위치적 맥락 확인 시작...")
    
    if state.get("image_part") is None:
        state["image_part"] = create_image_part(state["image_path"])
        if state["image_part"] is None:
            return {**state, "step1_result": {"error": "이미지 로드 실패"}}

    response_text, thinking_info = call_gemini_vision(model, STEP1_PROMPT, state["image_part"], "Step 1")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    print(f"✅ [Step 1] 완료: {result.get('location_type', 'unknown')}")
    
    return {
        **state,
        "step1_result": result
    }

def step2_spectral_analysis(state: AgentState) -> AgentState:
    """Step 2: 색채 스펙트럼 분석"""
    print("\n🎨 [Step 2] 색채 스펙트럼 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step2_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP2_PROMPT, state["image_part"], "Step 2")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    detected = result.get("cuprous_oxide_detected", False)
    print(f"✅ [Step 2] 완료: 아산화동 탐지 {'성공' if detected else '실패'}")
    
    return {
        **state,
        "step2_result": result
    }

def step3_thermal_gradient(state: AgentState) -> AgentState:
    """Step 3: 열적 구배 분석"""
    print("\n🔥 [Step 3] 열적 구배 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step3_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP3_PROMPT, state["image_part"], "Step 3")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    gradient_detected = result.get("thermal_gradient_detected", False)
    print(f"✅ [Step 3] 완료: 열적 구배 {'탐지됨' if gradient_detected else '미탐지'}")
    
    return {
        **state,
        "step3_result": result
    }

def step4_surface_analysis(state: AgentState) -> AgentState:
    """Step 4: 금속 표면 상태 분석"""
    print("\n🔬 [Step 4] 금속 표면 상태 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step4_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP4_PROMPT, state["image_part"], "Step 4")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    erosion_detected = result.get("electrical_erosion_detected", False)
    print(f"✅ [Step 4] 완료: 전기적 부식 {'탐지됨' if erosion_detected else '미탐지'}")
    
    return {
        **state,
        "step4_result": result
    }

def final_judgment(state: AgentState) -> AgentState:
    """최종 판정: 4단계 결과 종합 및 신뢰도 점수 계산"""
    print("\n⚖️ [Final Judgment] 최종 판정 시작...")
    
    step1 = state.get("step1_result", {}) or {}
    step2 = state.get("step2_result", {}) or {}
    step3 = state.get("step3_result", {}) or {}
    step4 = state.get("step4_result", {}) or {}
    
    # 각 단계별 점수 추출
    step1_score = step1.get("confidence", 0) if not step1.get("error") else 0
    step2_score = step2.get("confidence", 0) if not step2.get("error") else 0
    step3_score = step3.get("confidence", 0) if not step3.get("error") else 0
    step4_score = step4.get("confidence", 0) if not step4.get("error") else 0
    
    # 핵심 지표 확인
    is_connection_point = step1.get("is_connection_point", False)
    cuprous_oxide_detected = step2.get("cuprous_oxide_detected", False)
    thermal_gradient_detected = step3.get("thermal_gradient_detected", False)
    electrical_erosion_detected = step4.get("electrical_erosion_detected", False)
    
    # 신뢰도 점수 계산 (가중치 적용)
    base_score = 0
    
    # 핵심 지표 가중치
    if is_connection_point: base_score += 20
    if cuprous_oxide_detected: base_score += 30
    if thermal_gradient_detected: base_score += 25
    if electrical_erosion_detected: base_score += 15
    
    # 각 단계별 신뢰도 점수의 평균 반영 (10%)
    avg_confidence = (step1_score + step2_score + step3_score + step4_score) / 4
    base_score += avg_confidence * 0.1
    
    # 핵심 3가지가 모두 확인되면 90% 이상 보장
    if is_connection_point and cuprous_oxide_detected and thermal_gradient_detected:
        base_score = max(base_score, 90)
    
    final_score = min(100, max(0, int(base_score)))
    
    # 증거 수집
    evidence = []
    if is_connection_point:
        evidence.append({"step": 1, "evidence": "접속점 위치 확인", "details": step1.get("location_description", "")})
    if cuprous_oxide_detected:
        evidence.append({"step": 2, "evidence": "아산화동 탐지", "details": step2.get("location_of_oxidation", "")})
    if thermal_gradient_detected:
        evidence.append({"step": 3, "evidence": "열적 구배 패턴 확인", "details": step3.get("heat_source_location", "")})
    if electrical_erosion_detected:
        evidence.append({"step": 4, "evidence": "전기적 부식 확인", "details": step4.get("erosion_pattern", "")})
    
    # 분석 요약 생성
    summary_parts = [f"접촉불량 판정 신뢰도: {final_score}%"]
    summary_parts.append(f"✓ 접속점 위치 확인: {step1.get('location_type', 'unknown')}" if is_connection_point else "✗ 접속점 위치 미확인")
    summary_parts.append("✓ 아산화동(Cu₂O) 탐지됨" if cuprous_oxide_detected else "✗ 아산화동 미탐지")
    summary_parts.append(f"✓ 열적 구배 패턴 확인: {step3.get('gradient_pattern', 'unknown')}" if thermal_gradient_detected else "✗ 열적 구배 패턴 미확인")
    summary_parts.append(f"✓ 전기적 부식 확인: {step4.get('surface_texture', 'unknown')}" if electrical_erosion_detected else "✗ 전기적 부식 미확인")
    
    analysis_summary = "\n".join(summary_parts)
    print(f"✅ [Final Judgment] 완료: 신뢰도 {final_score}%")
    
    return {
        **state,
        "confidence_score": final_score,
        "analysis_summary": analysis_summary,
        "evidence": evidence
    }

In [7]:
def create_agent_graph():
    """접촉불량 판별 에이전트 그래프 생성"""
    workflow = StateGraph(AgentState)
    
    # 노드 추가
    workflow.add_node("step1_location", step1_location_context)
    workflow.add_node("step2_spectral", step2_spectral_analysis)
    workflow.add_node("step3_thermal", step3_thermal_gradient)
    workflow.add_node("step4_surface", step4_surface_analysis)
    workflow.add_node("final_judgment", final_judgment)
    
    # 엣지 연결 (순차 실행)
    workflow.set_entry_point("step1_location")
    workflow.add_edge("step1_location", "step2_spectral")
    workflow.add_edge("step2_spectral", "step3_thermal")
    workflow.add_edge("step3_thermal", "step4_surface")
    workflow.add_edge("step4_surface", "final_judgment")
    workflow.add_edge("final_judgment", END)
    
    return workflow.compile()

# 전역 그래프 객체
try:
    agent_app = create_agent_graph()
    print("✅ Langgraph StateGraph 구성 완료")
except Exception as e:
    print(f"⚠️ 그래프 구성 실패 (Langgraph 미설치 등): {e}")
    agent_app = None

def analyze_connection_failure(image_path: str) -> dict:
    """전체 접촉불량 분석 실행 함수"""
    if agent_app is None:
        return {"error": "Agent 그래프가 초기화되지 않았습니다."}

    initial_state: AgentState = {
        "image_path": image_path,
        "image_part": None,
        "step1_result": None,
        "step2_result": None,
        "step3_result": None,
        "step4_result": None,
        "confidence_score": 0,
        "analysis_summary": "",
        "evidence": []
    }
    
    print(f"\n{'='*60}\n🔍 접촉불량 분석 시작: {image_path}\n{'='*60}")
    
    try:
        final_state = agent_app.invoke(initial_state)
        return {
            "confidence_score": final_state["confidence_score"],
            "analysis_summary": final_state["analysis_summary"],
            "step1_result": final_state["step1_result"],
            "step2_result": final_state["step2_result"],
            "step3_result": final_state["step3_result"],
            "step4_result": final_state["step4_result"],
            "evidence": final_state["evidence"]
        }
    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        return {"error": str(e)}

✅ Langgraph StateGraph 구성 완료


In [8]:
def test_single_step(step_name: Literal["step1", "step2", "step3", "step4"], image_path: str, prev_state: Optional[AgentState] = None):
    """
    특정 단계만 독립적으로 테스트하기 위한 함수
    """
    print(f"\n🧪 [Test] {step_name} 독립 실행 테스트 중...")
    
    if prev_state:
        state = prev_state.copy()
    else:
        state: AgentState = {
            "image_path": image_path,
            "image_part": None, # 노드 내부에서 생성됨
            "step1_result": None,
            "step2_result": None,
            "step3_result": None,
            "step4_result": None,
            "confidence_score": 0,
            "analysis_summary": "",
            "evidence": []
        }
    
    try:
        if step_name == "step1":
            result_state = step1_location_context(state)
            print("결과:", json.dumps(result_state["step1_result"], indent=2, ensure_ascii=False))
        elif step_name == "step2":
            result_state = step2_spectral_analysis(state)
            print("결과:", json.dumps(result_state["step2_result"], indent=2, ensure_ascii=False))
        elif step_name == "step3":
            result_state = step3_thermal_gradient(state)
            print("결과:", json.dumps(result_state["step3_result"], indent=2, ensure_ascii=False))
        elif step_name == "step4":
            result_state = step4_surface_analysis(state)
            print("결과:", json.dumps(result_state["step4_result"], indent=2, ensure_ascii=False))
        return result_state
    except Exception as e:
        print(f"❌ 테스트 실패: {e}")
        return None

In [9]:
if __name__ == "__main__":
    # 테스트할 이미지 경로 설정 (노트북 기준 상대 경로)
    TEST_IMAGE_PATH = "../data/Cu2O_Breeding.jpg"
    
    # 이미지 파일 존재 여부 확인
    if not os.path.exists(TEST_IMAGE_PATH):
        print(f"⚠️ 경고: 테스트 이미지 '{TEST_IMAGE_PATH}'가 없습니다. 경로를 확인하세요.")
    else:
        # 전체 분석 실행
        result = analyze_connection_failure(TEST_IMAGE_PATH)
        
        # 결과 출력
        print("\n" + "="*60)
        print("📊 분석 결과")
        print("="*60)
        print(result.get("analysis_summary", ""))
        print(f"\n신뢰도 점수: {result.get('confidence_score', 0)}%")
        print("\n증거:")
        for ev in result.get("evidence", []):
            print(f"  - Step {ev.get('step')}: {ev.get('evidence')}")


🔍 접촉불량 분석 시작: ../data/Cu2O_Breeding.jpg

🔍 [Step 1] 위치적 맥락 확인 시작...
✅ [Step 1] 완료: splicing

🎨 [Step 2] 색채 스펙트럼 분석 시작...
✅ [Step 2] 완료: 아산화동 탐지 성공

🔥 [Step 3] 열적 구배 분석 시작...
✅ [Step 3] 완료: 열적 구배 탐지됨

🔬 [Step 4] 금속 표면 상태 분석 시작...
✅ [Step 4] 완료: 전기적 부식 탐지됨

⚖️ [Final Judgment] 최종 판정 시작...
✅ [Final Judgment] 완료: 신뢰도 98%

📊 분석 결과
접촉불량 판정 신뢰도: 98%
✓ 접속점 위치 확인: splicing
✓ 아산화동(Cu₂O) 탐지됨
✓ 열적 구배 패턴 확인: linear
✓ 전기적 부식 확인: rough

신뢰도 점수: 98%

증거:
  - Step 1: 접속점 위치 확인
  - Step 2: 아산화동 탐지
  - Step 3: 열적 구배 패턴 확인
  - Step 4: 전기적 부식 확인
